# Copyright 2022 Cognite AS

## Import the Libraries and Modules

In [1]:
%pip install pandas
%pip install cognite-sdk

import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Add utils directory to path (resolve from project root)
utils_path = Path(__file__).parent.parent / "utils" if '__file__' in globals() else Path("../utils").resolve()
if str(utils_path) not in sys.path:
    sys.path.insert(0, str(utils_path))

# Import and reload module to pick up changes (run this cell after modifying cognite_auth.py)
import importlib
import cognite_auth  # type: ignore
importlib.reload(cognite_auth)
from cognite_auth import interactive_client  # type: ignore

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Create the Cognite Client

In [2]:
customer = "oxy-aws-dev"
token_cache_path = Path.home() / ".cognite" / "token_cache" / f"{customer}.json"
client = interactive_client(customer, token_cache_path)

# Check Identity


In [4]:
# Get the logged-in user's profile
user_profile = client.iam.user_profiles.me()

# Print user information
print(f"User Identifier: {user_profile.user_identifier}")
print(f"Name: {user_profile.given_name} {user_profile.surname}")
print(f"Email: {user_profile.email}")
print(f"Full profile: {user_profile}")

print(token_cache_path)

User Identifier: kx-ut87k69sRB2pqrRzqZg
Name: Joe O'Bryant
Email: Joe_O'Bryant@oxy.com
Full profile: {
    "user_identifier": "kx-ut87k69sRB2pqrRzqZg",
    "last_updated_time": "2025-11-06 20:46:41.667+00:00",
    "given_name": "Joe",
    "surname": "O'Bryant",
    "email": "Joe_O'Bryant@oxy.com",
    "display_name": "O'Bryant, Joe (Cognite LLC)"
}
C:\Users\JoeO'Bryant\.cognite\token_cache\oxy-aws-dev.json


## Check Login Status By Running Assets List

In [5]:
# Get groups for all defined customer configurations
from cognite_auth import CUSTOMER_CONFIGS  # type: ignore

list_of_groups = {}
for customer_name in CUSTOMER_CONFIGS.keys():
    print(f"Fetching groups for customer: {customer_name}")
    try:
        # Create client for this customer
        customer_cache_path = Path.home() / ".cognite" / "token_cache" / f"{customer_name}.json"
        customer_client = interactive_client(customer_name, customer_cache_path)
        
        # Get groups for this customer
        groups = customer_client.iam.groups.list(all=True)
        list_of_groups[customer_name] = groups
        print(f"  ✓ Found {len(groups)} groups for {customer_name}")
    except Exception as e:
        print(f"  ✗ Error fetching groups for {customer_name}: {e}")
        list_of_groups[customer_name] = None

print(f"\nTotal customers processed: {len(list_of_groups)}")

Fetching groups for customer: oxy-oog-dev
  ✓ Found 8 groups for oxy-oog-dev
Fetching groups for customer: oxy-aws
  ✓ Found 37 groups for oxy-aws
Fetching groups for customer: oxy-aws-dev
  ✓ Found 39 groups for oxy-aws-dev

Total customers processed: 3


In [7]:
# Display raw capabilities from groups
import json

print("=" * 80)
print("RAW CAPABILITIES FROM GROUPS")
print("=" * 80)

for customer_name, groups in list_of_groups.items():
    if groups is None:
        print(f"\n{customer_name}: No groups (error occurred)")
        continue
    
    print(f"\n{'='*80}")
    print(f"Customer: {customer_name} ({len(groups)} groups)")
    print(f"{'='*80}")
    
    for i, group in enumerate(groups[:3], 1):  # Show first 3 groups per customer
        print(f"\n--- Group {i}: {getattr(group, 'name', 'Unknown')} ---")
        print(f"Group ID: {getattr(group, 'id', 'N/A')}")
        print(f"Source ID: {getattr(group, 'source_id', 'N/A')}")
        
        if hasattr(group, 'capabilities') and group.capabilities:
            print(f"\nCapabilities ({len(group.capabilities)} total):")
            for j, cap in enumerate(group.capabilities, 1):
                print(f"\n  Capability {j}:")
                print(f"    Type: {type(cap).__name__}")
                print(f"    Raw object: {cap}")
                print(f"    Attributes: {dir(cap)}")
                
                # Show all attributes and their values
                for attr in dir(cap):
                    if not attr.startswith('_'):
                        try:
                            value = getattr(cap, attr)
                            if not callable(value):
                                print(f"      {attr}: {value}")
                        except:
                            pass
        else:
            print("\nNo capabilities")
    
    if len(groups) > 3:
        print(f"\n... and {len(groups) - 3} more groups (showing first 3 only)")


RAW CAPABILITIES FROM GROUPS

Customer: oxy-oog-dev (8 groups)

--- Group 1: temp-cdf-extractor-oog-dev ---
Group ID: 2259313790107193
Source ID: 02f1c53d-59b5-48be-8b61-5c21ab1db42e

Capabilities (7 total):

  Capability 1:
    Type: TimeSeriesAcl
    Raw object: TimeSeriesAcl(actions=[<TimeSeriesAcl Action.Read: 'READ'>, <TimeSeriesAcl Action.Write: 'WRITE'>], scope=AllScope())
    Attributes: ['Action', 'Scope', '__abstractmethods__', '__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__post_init__', '__reduce__', '__reduce_ex__', '__replace__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_capabilit

In [8]:
# Capability extraction utilities (Single Responsibility: Extract capability strings from objects)

def extract_resource_name(capability) -> str:
    """Extract resource name from capability type (e.g., TimeSeriesAcl -> timeseries)."""
    type_name = type(capability).__name__
    # Remove 'Acl' suffix if present
    if type_name.endswith('Acl'):
        resource = type_name[:-3]
    else:
        resource = type_name
    
    # Convert CamelCase to lowercase (e.g., TimeSeries -> timeseries)
    import re
    # Insert underscores before capital letters, then lowercase
    resource = re.sub(r'(?<!^)(?=[A-Z])', '_', resource).lower()
    return resource


def extract_action_name(action) -> str:
    """Extract action name from action object (e.g., Action.Read -> read)."""
    # Try to get the action name from the action object's attributes
    if hasattr(action, 'name'):
        return str(action.name).lower()
    if hasattr(action, 'value'):
        return str(action.value).lower()
    
    # Fallback: parse from string representation
    action_str = str(action)
    # Extract action name from string like "TimeSeriesAcl Action.Read: 'READ'"
    # or "Action.Read" or "<Action.Read: 'READ'>"
    if 'Action.' in action_str:
        # Extract the part after "Action."
        parts = action_str.split('Action.')[-1]
        action_name = parts.split(':')[0].split("'")[0].split('>')[0].strip()
    elif '.' in action_str:
        # Fallback: get the part after the last dot
        action_name = action_str.split('.')[-1].split(':')[0].split("'")[0].split('>')[0].strip()
    else:
        action_name = action_str.split(':')[0].split("'")[0].split('>')[0].strip()
    
    return action_name.lower()


def is_all_scope(scope) -> bool:
    """Check if scope is AllScope."""
    if scope is None:
        return True  # None scope is treated as AllScope
    scope_type = type(scope).__name__
    if scope_type == 'AllScope':
        return True
    # Check if it's an AllScope instance by checking for 'all' attribute
    if hasattr(scope, 'all'):
        # AllScope typically has an 'all' attribute that is True or the scope itself
        return True
    return False


def extract_scope_string(scope) -> str | None:
    """Extract scope string from capability scope object. Returns None for AllScope."""
    if is_all_scope(scope):
        return None
    
    # Try to extract scope value from different scope types
    # Common scope types might have 'data_set_id', 'data_set_ids', 'project', etc.
    scope_parts = []
    
    # Check for common scope attributes
    for attr in ['data_set_id', 'data_set_ids', 'project', 'project_id', 'project_ids']:
        if hasattr(scope, attr):
            value = getattr(scope, attr)
            if value:
                if isinstance(value, (list, tuple)):
                    scope_parts.append(f"{attr}={','.join(str(v) for v in value)}")
                else:
                    scope_parts.append(f"{attr}={value}")
    
    # If no specific attributes found, use string representation
    if not scope_parts:
        scope_str = str(scope)
        # Remove common prefixes/suffixes
        if scope_str.startswith('Scope(') and scope_str.endswith(')'):
            scope_str = scope_str[6:-1]
        scope_parts.append(scope_str)
    
    return ':'.join(scope_parts) if scope_parts else None


def extract_capability_key(capability) -> str | None:
    """Extract a standardized capability key in format: resource:action or resource:action:scope."""
    if not hasattr(capability, 'actions') or not capability.actions:
        return None
    
    # Extract resource name
    resource = extract_resource_name(capability)
    
    # Extract and format actions
    action_names = []
    for action in capability.actions:
        action_name = extract_action_name(action)
        if action_name:
            action_names.append(action_name)
    
    if not action_names:
        return None
    
    # Sort actions for consistency
    action_names = sorted(set(action_names))
    
    # Build capability keys (one per action)
    capability_keys = []
    for action in action_names:
        cap_key = f"{resource}:{action}"
        
        # Add scope only if it's not AllScope
        if hasattr(capability, 'scope') and capability.scope:
            scope_str = extract_scope_string(capability.scope)
            if scope_str:
                # Append scope to the capability key
                cap_key = f"{cap_key}:{scope_str}"
        
        capability_keys.append(cap_key)
    
    # Return comma-separated if multiple actions, or single key
    return capability_keys if len(capability_keys) > 1 else capability_keys[0]


def collect_all_capabilities(groups_by_customer: dict) -> list[str]:
    """Collect all unique capabilities across all customers."""
    all_capabilities = set()
    
    for groups in groups_by_customer.values():
        if groups is None:
            continue
        
        for group in groups:
            if hasattr(group, 'capabilities') and group.capabilities:
                for cap in group.capabilities:
                    cap_keys = extract_capability_key(cap)
                    if cap_keys:
                        # Handle both single key (string) and multiple keys (list)
                        if isinstance(cap_keys, list):
                            for key in cap_keys:
                                all_capabilities.add(key)
                        else:
                            all_capabilities.add(cap_keys)
    
    return sorted(list(all_capabilities))


In [9]:
# DataFrame builder (Single Responsibility: Build DataFrame from groups data)

def parse_capability_key(cap_key: str) -> tuple[str, str, str | None]:
    """Parse a capability key into (resource, action, scope).
    
    Examples:
        'annotations:read' -> ('annotations', 'read', None)
        'annotations:read:data_set_id=123' -> ('annotations', 'read', 'data_set_id=123')
    """
    parts = cap_key.split(':')
    resource = parts[0]
    action = parts[1] if len(parts) > 1 else ''
    scope = ':'.join(parts[2:]) if len(parts) > 2 else None
    return (resource, action, scope)


def build_group_row(group, all_resources: set[str]) -> dict:
    """Build a single row dictionary for a group with resource-based capabilities."""
    row = {
        'Group Name': getattr(group, 'name', ''),
        'Group ID': getattr(group, 'id', ''),
        'Source ID': getattr(group, 'source_id', ''),
    }
    
    # Initialize all resource columns with empty sets
    resource_actions = {resource: set() for resource in all_resources}
    
    # Collect capabilities for this group
    if hasattr(group, 'capabilities') and group.capabilities:
        for cap_obj in group.capabilities:
            cap_keys = extract_capability_key(cap_obj)
            if cap_keys:
                # Handle both single key (string) and multiple keys (list)
                keys_list = cap_keys if isinstance(cap_keys, list) else [cap_keys]
                
                for key in keys_list:
                    resource, action, scope = parse_capability_key(key)
                    if resource in resource_actions:
                        # Include action (and scope if present) in the set
                        if scope:
                            resource_actions[resource].add(f"{action}:{scope}")
                        else:
                            resource_actions[resource].add(action)
    
    # Convert sets to comma-separated strings or "n/a"
    for resource in all_resources:
        actions_list = sorted(resource_actions[resource])
        row[resource] = ', '.join(actions_list) if actions_list else 'n/a'
    
    return row


def collect_all_resources(groups_by_customer: dict) -> set[str]:
    """Collect all unique resources across all customers."""
    all_resources = set()
    
    for groups in groups_by_customer.values():
        if groups is None:
            continue
        
        for group in groups:
            if hasattr(group, 'capabilities') and group.capabilities:
                for cap in group.capabilities:
                    cap_keys = extract_capability_key(cap)
                    if cap_keys:
                        keys_list = cap_keys if isinstance(cap_keys, list) else [cap_keys]
                        for key in keys_list:
                            resource, _, _ = parse_capability_key(key)
                            all_resources.add(resource)
    
    return all_resources


def build_customer_dataframe(groups, all_resources: set[str]) -> pd.DataFrame:
    """Build a DataFrame for a customer's groups with resource-based capability columns."""
    rows = [build_group_row(group, all_resources) for group in groups]
    df = pd.DataFrame(rows)
    
    # Reorder columns: Group info first, then resources
    group_cols = ['Group Name', 'Group ID', 'Source ID']
    resource_cols = sorted([r for r in all_resources if r in df.columns])
    
    # Filter out resource columns where all values are "n/a"
    resource_cols_with_data = []
    for col in resource_cols:
        if col in df.columns:
            # Check if any value in this column is not "n/a"
            if (df[col] != 'n/a').any():
                resource_cols_with_data.append(col)
    
    # Combine group columns with filtered resource columns
    df = df[group_cols + resource_cols_with_data]
    
    return df


In [10]:
# Excel writer (Single Responsibility: Write DataFrames to Excel)

def write_groups_to_excel(
    groups_by_customer: dict,
    all_resources: set[str],
    output_file: Path | str
) -> None:
    """Write groups data to Excel with one sheet per customer."""
    output_path = Path(output_file)
    
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        for customer_name, groups in groups_by_customer.items():
            sheet_name = customer_name[:31]  # Excel sheet name max 31 chars
            
            if groups is None:
                # Create empty sheet for failed customers
                pd.DataFrame({'Error': ['Failed to fetch groups']}).to_excel(
                    writer, sheet_name=sheet_name, index=False
                )
                continue
            
            # Build and save DataFrame for this customer
            df = build_customer_dataframe(groups, all_resources)
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"✓ Saved {len(df)} groups for {customer_name} to sheet '{sheet_name}'")
    
    print(f"\n✅ Excel file saved: {output_path.absolute()}")


In [11]:
# Main execution: Orchestrate the Excel export process
all_resources = collect_all_resources(list_of_groups)
output_file = Path("groups_by_customer.xlsx")
write_groups_to_excel(list_of_groups, all_resources, output_file)


✓ Saved 8 groups for oxy-oog-dev to sheet 'oxy-oog-dev'
✓ Saved 37 groups for oxy-aws to sheet 'oxy-aws'
✓ Saved 39 groups for oxy-aws-dev to sheet 'oxy-aws-dev'

✅ Excel file saved: c:\Users\JoeO'Bryant\OneDrive - Cognite AS\projects\cognite_python_sdk_training\using-cognite-python-sdk\notebooks\groups_by_customer.xlsx
